In [7]:

"""
Problem 2 — Insider Trading Signal (Equity, Bonus)
==================================================
Updated notebook version with:
- clearer console logs for screenshots
- stricter, more competition-friendly thresholds
- cleaner EDGAR event classification
- source_url fallback handling
- output compression so the final CSV is less noisy
- clean final notebook display (summary + preview tables)

Outputs: p2_signals.csv
"""

import time
import warnings
import numpy as np
import pandas as pd
import requests
from time import sleep
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Configuration ──────────────────────────────────────────────────────────────
DATA_DIR    = 'data_dir/equity'
OUTPUT_FILE = "p2_signals.csv"

EDGAR_URL   = "https://efts.sec.gov/LATEST/search-index"
EDGAR_START = "2026-01-01"
EDGAR_END   = "2026-02-20"

# Stricter thresholds to reduce noisy alerts
VOLUME_Z_THRESHOLD    = 3.0
DRIFT_SIGMA_THRESHOLD = 2.5
TRADER_ZSCORE_THRESH  = 3.0
BASELINE_DAYS         = 15

# Ranking / compression controls
MAX_SIGNALS_TOTAL     = 24
MAX_SIGNALS_PER_SECID = 2

EVENT_KEYWORDS = {
    "merger": [
        "merger", "acquisition", "acquired", "takeover", "combine",
        "definitive agreement", "strategic transaction", "business combination"
    ],
    "earnings": [
        "earnings", "revenue", "quarterly results", "guidance", "eps",
        "net income", "financial results", "quarter ended"
    ],
    "leadership": [
        "ceo", "chief executive", "resign", "appoint", "board",
        "director", "officer", "president", "chief financial officer", "cfo"
    ],
    "restatement": [
        "restate", "restatement", "correction", "material weakness",
        "error", "non-reliance"
    ],
    "bankruptcy": [
        "bankruptcy", "chapter 11", "restructuring", "insolvency", "liquidation"
    ],
    "contract": [
        "contract", "agreement", "partnership", "collaboration", "awarded",
        "license", "customer", "supply", "commercial"
    ],
}

# ── Utilities ──────────────────────────────────────────────────────────────────

def log(stage: str, msg: str):
    print(f"[{stage}] {msg}")

def safe_div(a, b):
    return a / b if b not in (0, 0.0) and pd.notna(b) else np.nan

# ── Data Loading ───────────────────────────────────────────────────────────────

def load_data(data_dir: str):
    t0 = time.time()
    log("LOAD", "Loading equity data")
    ohlcv  = pd.read_csv(f"{data_dir}/ohlcv.csv", parse_dates=["trade_date"])
    trades = pd.read_csv(f"{data_dir}/trade_data.csv", parse_dates=["timestamp"])

    log(
        "LOAD",
        f"ohlcv rows={len(ohlcv):,} | sec_ids={ohlcv['sec_id'].nunique()} | "
        f"dates={ohlcv['trade_date'].min().date()} to {ohlcv['trade_date'].max().date()}"
    )
    log(
        "LOAD",
        f"trade_data rows={len(trades):,} | traders={trades['trader_id'].nunique()}"
    )
    log("LOAD", f"Data loading time: {time.time()-t0:.2f} sec")
    return ohlcv, trades

def build_ticker_map(ohlcv: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in ["sec_id", "ticker", "name"] if c in ohlcv.columns]
    return ohlcv[cols].drop_duplicates().reset_index(drop=True)

# ── EDGAR ──────────────────────────────────────────────────────────────────────

def classify_event(text: str) -> str:
    t = str(text).lower()
    for event_type, keywords in EVENT_KEYWORDS.items():
        if any(kw in t for kw in keywords):
            return event_type
    return "other"

def build_search_url(ticker: str) -> str:
    return (
        f"{EDGAR_URL}?q=%22{ticker}%22&forms=8-K&dateRange=custom"
        f"&startdt={EDGAR_START}&enddt={EDGAR_END}"
    )

def fetch_8k_for_ticker(ticker: str, entity_name: str, start_date: str, end_date: str,
                        session: requests.Session) -> list:
    query_terms = [ticker]
    if isinstance(entity_name, str) and entity_name and entity_name.upper() != str(ticker).upper():
        query_terms.append(entity_name)

    all_rows = []

    for qterm in query_terms:
        params = {
            "q":         f'"{qterm}"',
            "forms":     "8-K",
            "dateRange": "custom",
            "startdt":   start_date,
            "enddt":     end_date,
        }
        try:
            resp = session.get(EDGAR_URL, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
        except Exception:
            continue

        hits = data.get("hits", {}).get("hits", [])
        for hit in hits:
            src = hit.get("_source", {})
            entity = src.get("entity_name", "") or entity_name or ticker
            file_date = src.get("file_date", "")
            file_path = src.get("file_path", "")
            form_type = src.get("form_type", "")
            display_names = src.get("display_names", [])

            headline_raw = entity
            if display_names:
                first = display_names[0]
                if isinstance(first, dict):
                    headline_raw = first.get("name", entity)
                else:
                    headline_raw = str(first)

            combined_text = " ".join([
                str(headline_raw),
                str(entity),
                str(src.get("period_of_report", "")),
                str(src.get("items", "")),
            ]).strip()

            all_rows.append({
                "ticker":     ticker,
                "entity":     entity,
                "file_date":  file_date,
                "form_type":  form_type,
                "headline":   headline_raw if headline_raw else entity,
                "raw_text":   combined_text,
                "filing_url": f"https://www.sec.gov{file_path}" if file_path else build_search_url(ticker),
            })

        if all_rows:
            break

    return all_rows

def scrape_edgar(ticker_map: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    t0 = time.time()
    session = requests.Session()
    session.headers.update({"User-Agent": "surveillance-hackathon contact@example.com"})

    tickers = ticker_map["ticker"].tolist()
    log("EDGAR", f"Scraping 8-K filings for {len(tickers)} tickers")

    all_results = []
    for i, row in ticker_map.iterrows():
        ticker = row["ticker"]
        entity_name = row["name"] if "name" in row and pd.notna(row["name"]) else ""
        results = fetch_8k_for_ticker(ticker, entity_name, start_date, end_date, session)
        if results:
            log("EDGAR", f"[{i+1:02d}/{len(ticker_map)}] {ticker}: {len(results)} filing rows")
        else:
            log("EDGAR", f"[{i+1:02d}/{len(ticker_map)}] {ticker}: 0 filing rows")
        all_results.extend(results)
        sleep(0.35)

    if not all_results:
        log("EDGAR", "No EDGAR results retrieved. Fallback path will be used.")
        return pd.DataFrame(columns=[
            "ticker", "entity", "file_date", "form_type", "headline",
            "raw_text", "filing_url", "event_type"
        ])

    filings = pd.DataFrame(all_results)
    filings["file_date"] = pd.to_datetime(filings["file_date"], errors="coerce")
    filings = filings.dropna(subset=["file_date"]).copy()
    filings["event_type"] = filings["raw_text"].apply(classify_event)
    filings["headline"] = filings["headline"].fillna("").astype(str).str.slice(0, 180)
    filings["source_url"] = filings["filing_url"].fillna("")

    # deduplicate by ticker + file_date + headline
    filings = filings.sort_values(["ticker", "file_date", "headline"]).drop_duplicates(
        subset=["ticker", "file_date", "headline"], keep="first"
    ).reset_index(drop=True)

    log("EDGAR", f"Total filings kept: {len(filings)}")
    log("EDGAR", f"Event type counts: {filings['event_type'].value_counts().to_dict()}")
    log("EDGAR", f"EDGAR scraping time: {time.time()-t0:.2f} sec")
    return filings

# ── Feature Engineering ────────────────────────────────────────────────────────

def build_ohlcv_baseline(ohlcv: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    df = ohlcv.sort_values(["sec_id", "trade_date"]).copy()
    df["close_ret"] = df.groupby("sec_id")["close"].pct_change()

    grp = df.groupby("sec_id")
    df["vol_15d_mean"] = grp["volume"].transform(
        lambda x: x.shift(1).rolling(BASELINE_DAYS, min_periods=5).mean()
    )
    df["vol_15d_std"] = grp["volume"].transform(
        lambda x: x.shift(1).rolling(BASELINE_DAYS, min_periods=5).std()
    )
    df["ret_15d_mean"] = grp["close_ret"].transform(
        lambda x: x.shift(1).rolling(BASELINE_DAYS, min_periods=5).mean()
    )
    df["ret_15d_std"] = grp["close_ret"].transform(
        lambda x: x.shift(1).rolling(BASELINE_DAYS, min_periods=5).std()
    )
    df["vol_z"] = (df["volume"] - df["vol_15d_mean"]) / df["vol_15d_std"].replace(0, np.nan)
    log("FEATURES", f"Built OHLCV baseline in {time.time()-t0:.2f} sec")
    return df

def compute_pre_drift(ohlcv_base: pd.DataFrame, sec_id: int, event_date: pd.Timestamp) -> dict:
    sec_data = ohlcv_base[ohlcv_base["sec_id"] == sec_id].copy()
    pre_window = sec_data[
        (sec_data["trade_date"] >= event_date - pd.Timedelta(days=10)) &
        (sec_data["trade_date"] < event_date)
    ].tail(5)

    if len(pre_window) < 2:
        return {
            "pre_drift": np.nan,
            "car_z": np.nan,
            "vol_z_max": np.nan,
            "drift_flag": 0,
            "vol_flag": 0,
            "window_start": None,
            "window_rows": 0
        }

    baseline_ret = pre_window["ret_15d_mean"].mean()
    baseline_std = pre_window["ret_15d_std"].mean()
    car = (pre_window["close_ret"] - baseline_ret).sum()
    car_z = car / (baseline_std * np.sqrt(len(pre_window))) if pd.notna(baseline_std) and baseline_std > 0 else 0
    vol_z_max = pre_window["vol_z"].max()

    drift_flag = int(abs(car_z) >= DRIFT_SIGMA_THRESHOLD)
    vol_flag = int(pd.notna(vol_z_max) and vol_z_max >= VOLUME_Z_THRESHOLD)

    return {
        "pre_drift": round(car, 6),
        "car_z": round(car_z, 3),
        "vol_z_max": round(vol_z_max, 3) if pd.notna(vol_z_max) else np.nan,
        "drift_flag": drift_flag,
        "vol_flag": vol_flag,
        "window_start": pre_window["trade_date"].min(),
        "window_rows": len(pre_window)
    }

def find_suspicious_traders(trades: pd.DataFrame, sec_id: int, event_date: pd.Timestamp) -> dict:
    filled = trades[
        (trades["sec_id"] == sec_id) &
        (trades["order_status"] == "FILLED")
    ].copy()

    if filled.empty:
        return {}

    filled["trade_date"] = filled["timestamp"].dt.normalize()

    pre_start = event_date - pd.Timedelta(days=5)
    pre_end = event_date - pd.Timedelta(days=1)
    pre_filled = filled[(filled["trade_date"] >= pre_start) & (filled["trade_date"] <= pre_end)].copy()
    if pre_filled.empty:
        return {}

    baseline = (
        filled.groupby(["trader_id", "trade_date"])["quantity"].sum().reset_index()
        .groupby("trader_id")["quantity"].agg(["mean", "std"])
        .rename(columns={"mean": "baseline_mean", "std": "baseline_std"})
        .reset_index()
    )
    pre_daily = pre_filled.groupby(["trader_id", "trade_date", "side"])["quantity"].sum().reset_index()
    pre_daily = pre_daily.merge(baseline, on="trader_id", how="left")
    pre_daily["qty_z"] = (
        (pre_daily["quantity"] - pre_daily["baseline_mean"]) /
        pre_daily["baseline_std"].replace(0, np.nan)
    )

    suspicious = pre_daily[pre_daily["qty_z"] >= TRADER_ZSCORE_THRESH].copy()
    if suspicious.empty:
        return {}

    result = {}
    for trader_id, grp in suspicious.groupby("trader_id"):
        best = grp.sort_values(["qty_z", "quantity"], ascending=False).iloc[0]
        result[trader_id] = {
            "max_qty": int(best["quantity"]),
            "max_qty_z": round(best["qty_z"], 2),
            "side": str(best["side"]),
            "trade_dates": sorted(pd.to_datetime(grp["trade_date"]).dt.strftime("%Y-%m-%d").unique().tolist()),
        }
    return result

# ── Signal Building ────────────────────────────────────────────────────────────

def signal_score(row):
    score = 0.0
    score += max(0, row.get("vol_z_max", 0) or 0) * 1.2
    score += abs(row.get("car_z", 0) or 0) * 1.0
    score += row.get("num_suspicious_traders", 0) * 1.5
    if row.get("event_type") == "merger":
        score += 2.0
    elif row.get("event_type") in {"earnings", "leadership", "restatement", "bankruptcy", "contract"}:
        score += 1.0
    return round(score, 3)

def build_signals(filings: pd.DataFrame, ohlcv: pd.DataFrame, trades: pd.DataFrame,
                  ticker_map: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    ohlcv_base = build_ohlcv_baseline(ohlcv)
    tm_dict = dict(zip(ticker_map["ticker"], ticker_map["sec_id"]))

    if filings.empty:
        log("DETECT", "No EDGAR filings. Using fallback volume-only path.")
        return build_volume_only_signals(ohlcv_base, trades, ticker_map)

    rows = []
    seen = set()
    raw_candidates = 0

    for _, filing in filings.iterrows():
        ticker = filing["ticker"]
        event_date = filing["file_date"]
        sec_id = tm_dict.get(ticker)

        if sec_id is None or not isinstance(event_date, pd.Timestamp):
            continue

        key = (sec_id, pd.Timestamp(event_date).normalize())
        if key in seen:
            continue
        seen.add(key)

        raw_candidates += 1
        drift = compute_pre_drift(ohlcv_base, sec_id, pd.Timestamp(event_date).normalize())
        susp = find_suspicious_traders(trades, sec_id, pd.Timestamp(event_date).normalize())

        pre_drift_flag = int(max(drift["drift_flag"], drift["vol_flag"], 1 if len(susp) > 0 else 0))

        remarks_parts = []
        if drift["vol_flag"]:
            remarks_parts.append(
                f"Volume z-score reached {drift['vol_z_max']:.1f}σ in the T-5 to T-1 window."
            )
        if drift["drift_flag"]:
            remarks_parts.append(
                f"Cumulative abnormal return was {drift['pre_drift']:.4f} with drift strength {drift['car_z']:.1f}σ."
            )
        if susp:
            top_traders = list(susp.items())[:2]
            for tid, info in top_traders:
                remarks_parts.append(
                    f"Trader {tid} showed unusual {info['side']} activity: {info['max_qty']:,} shares, qty z-score {info['max_qty_z']:.1f}σ."
                )
        if not remarks_parts:
            remarks_parts.append("No strong pre-announcement trading signal detected from the 15-day baseline and trader activity checks.")

        rows.append({
            "sec_id": sec_id,
            "ticker": ticker,
            "event_date": pd.Timestamp(event_date).strftime("%Y-%m-%d"),
            "event_type": filing["event_type"],
            "headline": str(filing["headline"])[:200],
            "source_url": filing["source_url"] if "source_url" in filing else filing.get("filing_url", ""),
            "pre_drift_flag": pre_drift_flag,
            "suspicious_window_start": drift["window_start"].strftime("%Y-%m-%d") if drift["window_start"] is not None else "",
            "remarks": " ".join(remarks_parts),
            "vol_z_max": drift["vol_z_max"],
            "car_z": drift["car_z"],
            "num_suspicious_traders": len(susp),
        })

    signals = pd.DataFrame(rows)
    log("DETECT", f"Raw filing-linked candidates evaluated: {raw_candidates}")

    if signals.empty:
        return signals

    signals["signal_score"] = signals.apply(signal_score, axis=1)

    # keep only stronger signals:
    strong_mask = (
        (signals["event_type"] != "other") |
        (signals["pre_drift_flag"] == 1)
    )
    signals = signals[strong_mask].copy()

    # rank and compress
    signals = signals.sort_values(
        ["pre_drift_flag", "signal_score", "vol_z_max", "car_z"],
        ascending=[False, False, False, False]
    ).copy()

    signals = signals.groupby("sec_id", group_keys=False).head(MAX_SIGNALS_PER_SECID).copy()
    signals = signals.head(MAX_SIGNALS_TOTAL).reset_index(drop=True)

    log("DETECT", f"Signals after filtering/compression: {len(signals)}")
    log("DETECT", f"Signal building time: {time.time()-t0:.2f} sec")
    return signals

def build_volume_only_signals(ohlcv_base: pd.DataFrame, trades: pd.DataFrame,
                              ticker_map: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    event_week = ohlcv_base[ohlcv_base["trade_date"] >= "2026-02-17"].copy()
    high_vol = event_week[event_week["vol_z"] >= VOLUME_Z_THRESHOLD].copy()

    ticker_dict = dict(zip(ticker_map["sec_id"], ticker_map["ticker"]))
    rows = []
    for _, row in high_vol.iterrows():
        sec_id = row["sec_id"]
        event_date = pd.Timestamp(row["trade_date"]).normalize()
        susp = find_suspicious_traders(trades, sec_id, event_date + pd.Timedelta(days=1))

        remarks = (
            f"Volume was {row['volume']:,} shares on {event_date.date()}, "
            f"which is {row['vol_z']:.1f}σ above the 15-day baseline."
        )
        if susp:
            top_tid, info = sorted(susp.items(), key=lambda x: x[1]["max_qty_z"], reverse=True)[0]
            remarks += (
                f" Trader {top_tid} also showed unusual {info['side']} activity "
                f"with {info['max_qty']:,} shares and qty z-score {info['max_qty_z']:.1f}σ."
            )

        rows.append({
            "sec_id": sec_id,
            "ticker": ticker_dict.get(sec_id, str(sec_id)),
            "event_date": event_date.strftime("%Y-%m-%d"),
            "event_type": "abnormal_volume",
            "headline": f"{ticker_dict.get(sec_id, sec_id)} abnormal volume event",
            "source_url": "",
            "pre_drift_flag": int(row["vol_z"] >= VOLUME_Z_THRESHOLD),
            "suspicious_window_start": event_date.strftime("%Y-%m-%d"),
            "remarks": remarks,
            "vol_z_max": round(row["vol_z"], 3),
            "car_z": np.nan,
            "num_suspicious_traders": len(susp),
        })

    signals = pd.DataFrame(rows)
    if not signals.empty:
        signals["signal_score"] = signals.apply(signal_score, axis=1)
        signals = signals.sort_values(
            ["pre_drift_flag", "signal_score", "vol_z_max"],
            ascending=[False, False, False]
        ).groupby("sec_id", group_keys=False).head(MAX_SIGNALS_PER_SECID).head(MAX_SIGNALS_TOTAL).reset_index(drop=True)

    log("DETECT", f"Fallback volume-only candidates kept: {len(signals)}")
    log("DETECT", f"Fallback signal build time: {time.time()-t0:.2f} sec")
    return signals

# ── Run ────────────────────────────────────────────────────────────────────────

def run(data_dir: str = DATA_DIR, output_file: str = OUTPUT_FILE):
    total_t0 = time.time()
    log("START", "Problem 2 pipeline started")

    ohlcv, trades = load_data(data_dir)
    ticker_map = build_ticker_map(ohlcv)
    log("MAP", f"Ticker map rows={len(ticker_map)}")

    filings = scrape_edgar(ticker_map, EDGAR_START, EDGAR_END)
    signals = build_signals(filings, ohlcv, trades, ticker_map)

    elapsed = time.time() - total_t0

    if signals.empty:
        log("FINAL", "No signals generated")
        return pd.DataFrame()

    signals["time_to_run"] = round(elapsed, 2)

    final_cols = [
        "sec_id", "event_date", "event_type", "headline", "source_url",
        "pre_drift_flag", "suspicious_window_start", "remarks", "time_to_run"
    ]
    signals = signals[[c for c in final_cols if c in signals.columns]].copy()
    signals.to_csv(output_file, index=False)

    flagged_count = int((signals["pre_drift_flag"] == 1).sum())

    log("FINAL", f"Output file: {output_file}")
    log("FINAL", f"Final signal rows written: {len(signals)}")
    log("FINAL", f"Flagged pre-drift rows: {flagged_count}")
    log("FINAL", f"Total runtime: {elapsed:.2f} sec")

    print("\n" + "="*80)
    print("FINAL OUTPUT SUMMARY")
    print("="*80)
    summary_df = pd.DataFrame({
        "metric": [
            "tickers_processed",
            "filings_retrieved",
            "final_signal_rows",
            "flagged_pre_drift_rows",
            "total_runtime_sec"
        ],
        "value": [
            ticker_map["ticker"].nunique(),
            len(filings),
            len(signals),
            flagged_count,
            round(elapsed, 2)
        ]
    })
    display(summary_df)

    print("\nTOP SIGNAL PREVIEW")
    preview_cols = ["sec_id", "event_date", "event_type", "pre_drift_flag"]
    display(signals[preview_cols].head(12))

    print("\nEXAMPLE REMARKS PREVIEW")
    remarks_preview = signals[["sec_id", "event_date", "remarks"]].head(5).copy()
    remarks_preview["remarks"] = remarks_preview["remarks"].str.slice(0, 180)
    display(remarks_preview)

    return signals

# Execute
signals_df = run()


[START] Problem 2 pipeline started
[LOAD] Loading equity data
[LOAD] ohlcv rows=532 | sec_ids=28 | dates=2026-01-26 to 2026-02-20
[LOAD] trade_data rows=3,036 | traders=6
[LOAD] Data loading time: 0.04 sec
[MAP] Ticker map rows=28
[EDGAR] Scraping 8-K filings for 28 tickers
[EDGAR] [01/28] AZN: 4 filing rows
[EDGAR] [02/28] HWM: 5 filing rows
[EDGAR] [03/28] AEE: 6 filing rows
[EDGAR] [04/28] ADI: 16 filing rows
[EDGAR] [05/28] ACLS: 9 filing rows
[EDGAR] [06/28] CTAS: 9 filing rows
[EDGAR] [07/28] C: 100 filing rows
[EDGAR] [08/28] DE: 100 filing rows
[EDGAR] [09/28] XOM: 6 filing rows
[EDGAR] [10/28] FDX: 5 filing rows
[EDGAR] [11/28] LMT: 3 filing rows
[EDGAR] [12/28] MSI: 8 filing rows
[EDGAR] [13/28] RSG: 9 filing rows
[EDGAR] [14/28] TGT: 7 filing rows
[EDGAR] [15/28] TXN: 10 filing rows
[EDGAR] [16/28] CRH: 6 filing rows
[EDGAR] [17/28] AVAV: 1 filing rows
[EDGAR] [18/28] IWM: 1 filing rows
[EDGAR] [19/28] IVW: 0 filing rows
[EDGAR] [20/28] SPYG: 0 filing rows
[EDGAR] [21/28] FA

,metric,value
0,tickers_processed,28.00
1,filings_retrieved,350.00
2,final_signal_rows,19.00
3,flagged_pre_drift_rows,16.00
4,total_runtime_sec,42.66



TOP SIGNAL PREVIEW


,sec_id,event_date,event_type,pre_drift_flag
0,35701,2026-02-20,other,1
1,4428506,2026-02-17,other,1
2,35701,2026-02-05,other,1
3,33771,2026-02-06,other,1
4,34926,2026-02-09,other,1
5,43205,2026-02-05,other,1
6,34926,2026-02-10,other,1
7,40039,2026-02-05,other,1
8,4428506,2026-02-10,merger,1
9,8126151,2026-02-17,other,1



EXAMPLE REMARKS PREVIEW


,sec_id,event_date,remarks
0,35701,2026-02-20,Volume z-score reached 8.1σ in the T-5 to T-1 window.
1,4428506,2026-02-17,Volume z-score reached 7.2σ in the T-5 to T-1 window.
2,35701,2026-02-05,Volume z-score reached 5.8σ in the T-5 to T-1 window.
3,33771,2026-02-06,Volume z-score reached 6.1σ in the T-5 to T-1 window.
4,34926,2026-02-09,Volume z-score reached 4.2σ in the T-5 to T-1 window. Cumulative abnormal re...
